In [3]:
import numpy as np
from collections import Counter

# class Node
- What feature was this node divided with?
- What was the division threshold on this node?
- What is the left node that we are pointing to?
- What is the right node that we are pointing to?
- What is the value for this node? 
    - If not None, then it is the leaf node.

In [17]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None): # * to enforce value as a keyword arg
        self.feature = feature # which feature this was divided with
        self.threshold = threshold # which threshold this was divided with 
        self.left = left # the left node that we are pointing to
        self.right = right # the right node that we are pointing to
        self.value = value # the value of the node, if is a leaf node

    def is_leaf_node(self):
        return self.value is not None

# class DecisionTree
- define the stopping criteria
    - **min_sample_split**: min number of samples that a node can have.
    - **max_depth**: How many layers of node?
    - **min impurity decrease**: min entropy change that needs to take place for a split to happen
    - **n_features**: a way to add randomness, especially for a random forest
- the **fit** method
    - grow a tree
    
- the **predict** method

In [20]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        #stopping criteria
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features # get it at the time of defining the tree
        self.root = None # access to the root of the tree in the inference time

    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features) # make sure that the n_features does not exceed the actual no. of features in the dataset
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # check the stopping criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # find the best split
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)
        best_features, best_threshold = self._best_split(X, y, feat_idxs)

        # create child node
        left_idxs, right_idxs = self._split(X[:, best_features], best_threshold)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth+1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth+1)

        return Node(best_features, best_threshold, left, right)


    def _most_common_label(self, y):
        counter = Counter(y)
        value = counter.most_common(1)[0][0]
        return value


    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                # calculate info gain
                gain = self._info_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold


    def _info_gain(self, y, X_column, threshold):
        # parent entropy
        parent_entropy = self._entropy(y)

        # create children
        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0 

        # claculate the weighted avg. entropy of children
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        #claculate the IG
        info_gain = parent_entropy - child_entropy
        return info_gain



    def _entropy(self, y):
        hist = np.bincount(y) # is it like a histogram, showing from 0 to max(y) how many times each value occurs in y
        proba = hist / len(y)
        return -np.sum([p * np.log(p) for p in proba if p>0])


    def _split(self, X_column, split_thr):
        left_idxs = np.argwhere(X_column <= split_thr).flatten()
        right_idxs = np.argwhere(X_column > split_thr).flatten()
        return left_idxs, right_idxs


    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])


    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)

        return self._traverse_tree(x, node.right)

# Test

In [22]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split


data = load_breast_cancer()

X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

clf = DecisionTree(max_depth=10)

clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

def accuracy(y_test, y_pred):
    return np.sum(y_test == y_pred) / len(y_test)

acc = accuracy(y_test, predictions)
print(acc)

0.9298245614035088
